<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Munsell_to_Lab_RGB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Munsell codes → RGB, CIE Lab, and DOCX export with colour rectangles

This notebook unifies two workflows and now includes **Bradford Von Kries chromatic adaptation** (Illuminant C → D65):

1. **Reference conversion**: Read `MilkMunsellCodes.csv` directly from GitHub (raw URL) to compute RGB and CIE L*a*b* from Munsell codes, then export `MilkMunsell_RGB_Lab.csv`.
2. **Manual upload**: Upload a custom CSV of Munsell codes (same column structure) and run the same conversion.

A single **mode switch** (`INPUT_MODE`) lets you choose which of the two data sources feeds the rest of the notebook. The default is `reference` (GitHub CSV); set it to `manual` to instead prompt for a manual file upload.

For both, it also:

- Chromatically adapts Munsell/Illuminant-C tristimulus values (XYZ_C) to Illuminant D65 using the **Bradford transform** within the **Von Kries** adaptation family, before computing CIE L*a*b* and sRGB — matching the sRGB/D65 reference white used downstream.
- Generates rectangular colour swatch images (via Pillow).
- Builds DOCX tables with the colour swatches inserted in the last column.
- Includes explicit RGB (0–255) columns `R`, `G`, `B` in the DOCX tables.
- Saves DOCX files that you can download from Colab's file browser or via `files.download(...)`.

Output data structure (columns, CSV names, DOCX table layout) is unchanged from the previous version of this notebook.

In [31]:
#@title Imports and dependency check
import sys
import subprocess

def ensure_package(pkg_name, pip_name=None):
    """Import a package, installing it via pip if missing."""
    if pip_name is None:
        pip_name = pkg_name
    try:
        __import__(pkg_name)
        print(f"Package {pkg_name} already available.")
    except ImportError:
        print(f"Package {pkg_name} not found, installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        __import__(pkg_name)
        print(f"Package {pkg_name} installed and imported.")

# Ensure required packages
ensure_package("colour", "colour-science")
ensure_package("docx", "python-docx")
ensure_package("PIL", "pillow")
ensure_package("pandas", "pandas")
ensure_package("numpy", "numpy")

# Now import them normally
import colour
import numpy as np
import pandas as pd
from PIL import Image
from docx import Document
from docx.shared import Inches
import os

# For Colab downloads (optional)
try:
    from google.colab import files
except ImportError:
    files = None

os.makedirs("output", exist_ok=True)

# Reference illuminants (CIE 1931 2-degree standard observer).
# Munsell renotation data is defined under Illuminant C; downstream Lab/sRGB
# calculations use Illuminant D65, so a chromatic-adaptation step is required
# to move tristimulus values from the C white point to the D65 white point.
illuminant_C = colour.CCS_ILLUMINANTS["CIE 1931 2 Degree Standard Observer"]["C"]
illuminant_D65 = colour.CCS_ILLUMINANTS["CIE 1931 2 Degree Standard Observer"]["D65"]

# White-point tristimulus values (Y = 1) required by the Von Kries adaptation routine.
WHITE_C_XYZ = colour.xy_to_XYZ(illuminant_C)
WHITE_D65_XYZ = colour.xy_to_XYZ(illuminant_D65)

# Chromatic adaptation transform used throughout this notebook: Bradford,
# applied within the Von Kries adaptation family.
CHROMATIC_ADAPTATION_TRANSFORM = "Bradford"

Package colour already available.
Package docx already available.
Package PIL already available.
Package pandas already available.
Package numpy already available.


In [32]:
#@title Conversion functions (with Bradford Von Kries adaptation)
def munsell_to_rgb_lab(code: str):
    """
    Convert a Munsell code string to XYZ, Lab (D65), sRGB, and RGB255.

    Procedure:
      1. Look up the Munsell renotation as xyY, referenced to Illuminant C.
      2. Convert xyY(C) to XYZ(C).
      3. Chromatically adapt XYZ(C) to XYZ(D65) using the Bradford transform
         within the Von Kries adaptation family, so that the white point
         matches the D65 reference used for Lab and sRGB.
      4. Convert the adapted XYZ(D65) to CIE Lab under D65.
      5. Convert the adapted XYZ(D65) to sRGB (clipped to [0, 1]) and to
         8-bit RGB255 values.
    """
    # Step 1-2: Munsell code -> xyY(C) -> XYZ(C)
    xyY = colour.notation.munsell_colour_to_xyY(code)
    XYZ_C = colour.xyY_to_XYZ(xyY)

    # Step 3: Bradford Von Kries chromatic adaptation, Illuminant C -> D65
    XYZ = colour.adaptation.chromatic_adaptation_VonKries(
        XYZ_C,
        WHITE_C_XYZ,
        WHITE_D65_XYZ,
        transform=CHROMATIC_ADAPTATION_TRANSFORM,
    )

    # Step 4: adapted XYZ(D65) -> CIE Lab(D65)
    Lab = colour.XYZ_to_Lab(XYZ, illuminant=illuminant_D65)

    # Step 5: adapted XYZ(D65) -> sRGB (0-1) -> RGB255
    RGB = np.clip(colour.XYZ_to_sRGB(XYZ), 0, 1)
    RGB255 = np.round(RGB * 255).astype(int)
    return XYZ, Lab, RGB, RGB255

def convert_munsell_dataframe(df_codes: pd.DataFrame) -> pd.DataFrame:
    """
    Given a DataFrame with columns ['Item', 'Munsell code'],
    return full RGB/Lab table (Bradford Von Kries D65-adapted).

    Output columns are unchanged from the previous version of this notebook:
    Item, Munsell code, L*, a*, b*, R, G, B, R_norm, G_norm, B_norm.
    """
    rows = []
    for _, row in df_codes.iterrows():
        code = str(row["Munsell code"]).strip()
        XYZ, Lab, RGB, RGB255 = munsell_to_rgb_lab(code)
        rows.append({
            "Item": row["Item"],
            "Munsell code": code,
            "L*": Lab[0],
            "a*": Lab[1],
            "b*": Lab[2],
            "R": RGB255[0],
            "G": RGB255[1],
            "B": RGB255[2],
            "R_norm": RGB[0],
            "G_norm": RGB[1],
            "B_norm": RGB[2],
        })
    return pd.DataFrame(rows)

In [33]:
#@title DOCX generation with RGB columns
def add_colour_table_docx(df: pd.DataFrame, docx_path: str, title: str):
    """
    Create swatch images and a DOCX table with rectangles in the last column.
    Columns: Item, Munsell code, L*, a*, b*, R, G, B, Colour sample.

    Lab and RGB values in `df` already reflect Bradford Von Kries adaptation
    from Illuminant C to D65 (performed upstream in convert_munsell_dataframe).
    """
    # Step 1: render one colour-swatch PNG per row, from the RGB255 columns.
    swatch_paths = []
    base_name = os.path.splitext(os.path.basename(docx_path))[0]
    for i, r in df.iterrows():
        rgb = (int(r["R"]), int(r["G"]), int(r["B"]))
        img = Image.new("RGB", (200, 80), rgb)
        path = os.path.join("output", f"swatch_{base_name}_{i+1}.png")
        img.save(path)
        swatch_paths.append(path)

    # Step 2: build the DOCX document and its header row.
    doc = Document()
    doc.add_heading(title, level=1)

    cols = 9  # Item, Munsell, L*, a*, b*, R, G, B, Colour sample
    table = doc.add_table(rows=1, cols=cols)
    hdr = table.rows[0].cells
    hdr[0].text = "Item"
    hdr[1].text = "Munsell code"
    hdr[2].text = "L*"
    hdr[3].text = "a*"
    hdr[4].text = "b*"
    hdr[5].text = "R"
    hdr[6].text = "G"
    hdr[7].text = "B"
    hdr[8].text = "Colour sample"

    # Step 3: populate one table row per sample, embedding the swatch image.
    for i, r in df.iterrows():
        cells = table.add_row().cells
        cells[0].text = str(r["Item"])
        cells[1].text = str(r["Munsell code"])
        cells[2].text = f"{r['L*']:.3f}"
        cells[3].text = f"{r['a*']:.3f}"
        cells[4].text = f"{r['b*']:.3f}"
        cells[5].text = str(int(r["R"]))
        cells[6].text = str(int(r["G"]))
        cells[7].text = str(int(r["B"]))
        run = cells[8].paragraphs[0].add_run()
        run.add_picture(swatch_paths[i], width=Inches(1.2), height=Inches(0.5))

    # Step 4: save the document and offer a Colab download if applicable.
    doc.save(docx_path)
    print(f"DOCX saved to: {docx_path}")
    if files is not None:
        files.download(docx_path)

In [34]:
#@title Input-mode switch: choose data source ("reference" or "manual")
# INPUT_MODE selects which workflow feeds the rest of the notebook:
#   "reference" -> read MilkMunsellCodes.csv directly from GitHub (default)
#   "manual"    -> prompt for a manual CSV upload (same column structure)
INPUT_MODE = "reference"  #@param ["reference", "manual"]

assert INPUT_MODE in ("reference", "manual"), \
    'INPUT_MODE must be either "reference" or "manual".'
print(f"Selected input mode: {INPUT_MODE}")

Selected input mode: reference


In [35]:
#@title Load data according to INPUT_MODE and run conversion
# Step 1: obtain df_codes with columns ['Item', 'Munsell code'] from the
# selected source, mirroring the two original workflows exactly.
if INPUT_MODE == "reference":
    # Reference conversion: read MilkMunsellCodes.csv directly from GitHub raw URL
    url = "https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/MilkMunsellCodes.csv"
    df_codes = pd.read_csv(url)
    df_codes.columns = ["Item", "Munsell code"]
    df_codes["Munsell code"] = df_codes["Munsell code"].astype(str).str.strip()
else:
    # Manual upload: prompt for a CSV with the same column structure.
    if files is not None:
        uploaded = files.upload()
        fname = next(iter(uploaded.keys()))
    else:
        fname = "YourManualMunsellCodes.csv"  # adjust if running locally
    df_codes = pd.read_csv(fname)
    df_codes.columns = ["Item", "Munsell code"]
    df_codes["Munsell code"] = df_codes["Munsell code"].astype(str).str.strip()

# Step 2: run the shared Bradford Von Kries conversion pipeline.
df_result = convert_munsell_dataframe(df_codes)

# Step 3: preserve original output filenames/variable names depending on mode,
# so downstream behaviour and file outputs match the previous notebook exactly.
if INPUT_MODE == "reference":
    df_ref = df_result
    df_ref.to_csv("MilkMunsell_RGB_Lab.csv", index=False)
    df_result_display = df_ref
else:
    df_manual = df_result
    out_csv_manual = "Manual_Munsell_RGB_Lab_v3_nohex.csv"
    df_manual.to_csv(out_csv_manual, index=False)
    df_result_display = df_manual

df_result_display

,Item,Munsell code,L*,a*,b*,R,G,B,R_norm,G_norm,B_norm
0,L6,5PB 7/4,70.829408,-0.452476,-14.358265,158,175,199,0.620205,0.684934,0.780931
1,L4,10PB 7/5.5,70.790233,8.874328,-19.540750,172,169,209,0.673762,0.664580,0.818177
2,L3,5P 7/4,70.807134,10.212741,-11.308607,183,168,194,0.716630,0.658595,0.760353
3,L2,10P 7/8,70.742190,29.647686,-13.966695,215,154,199,0.843190,0.605433,0.780742
4,LP6,10B 9/2,90.179613,-3.723640,-5.501962,214,229,237,0.838373,0.899950,0.929961
5,LP5,2.5PB 9/2,90.173982,-1.681442,-5.569462,218,228,237,0.855154,0.895170,0.930659
6,LP4,7.5PB 9/2,90.167202,1.001291,-5.343124,224,227,237,0.877977,0.888684,0.929247
7,LP3,5P 9/2,90.163255,3.330672,-4.223895,230,225,235,0.900746,0.882677,0.921203
8,LP2,2.5RP 9/2,90.164805,5.138305,-1.150460,236,224,229,0.925595,0.877323,0.898669
9,LP1,5RP 9/3,90.152064,10.331431,-0.266748,247,220,228,0.967614,0.863794,0.892704


In [36]:
#@title DOCX export (branches on INPUT_MODE, same output form as before)
if INPUT_MODE == "reference":
    docx_ref_path = os.path.join("output", "MilkColours_reference.docx")
    add_colour_table_docx(
        df_ref, docx_ref_path,
        "Milk sample colours (reference conversion, Bradford Von Kries D65, no hex)"
    )
else:
    docx_manual_path = os.path.join("output", "MilkColours_manual_v3_nohex.docx")
    add_colour_table_docx(
        df_manual, docx_manual_path,
        "Milk sample colours (manual upload conversion, Bradford Von Kries D65, no hex)"
    )

DOCX saved to: output/MilkColours_reference.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>